S3 with boto3

In [1]:
import boto3

In [2]:
s3 =  boto3.client('s3')

bucket_name = 'somerandomkgptalkie2'

In [3]:
response = s3.list_buckets()

In [ ]:
response

In [8]:
##test
response['Buckets'][0]['Name']

'youtube786786'

In [9]:
def create_bucket(bucket_name):
    s3.create_bucket(Bucket=bucket_name)
    print("Bucket is created")

create_bucket(bucket_name)

Bucket is created


Upload single file

In [10]:
import os

In [ ]:
os.getcwd()

In [14]:
os.chdir('..')

In [22]:
def upload_file(file_path, object_name=None):
    if object_name is None:
        object_name = os.path.basename(file_path)
    
    s3.upload_file(file_path, bucket_name, object_name)
    print(f"File {object_name} is uploaded to {bucket_name}")


In [23]:
upload_file('data/index.html')

File index.html is uploaded to somerandomkgptalkie2


In [21]:
upload_file('data/index.html', 'randomeName.html')

List all object in bucket

In [25]:
def list_objects():
    response= s3.list_objects_v2(Bucket=bucket_name)
    for obj in response['Contents']:
        print(obj['Key'])

list_objects()

index.html
randomeName.html


Dowload s3 file to local system

In [30]:
def download_file(object_name, file_path):
    if not os.path.exists(os.path.dirname(file_path)):
        os.makedirs(os.path.dirname(file_path))
    
    s3.download_file(bucket_name, object_name, file_path)
    print(f"File {object_name} is downloaded to {file_path}")

download_file('index.html', 'data_download/index.html')

File index.html is downloaded to data_download/index.html


Upload a folder to S3

In [ ]:
def upload_directory(directory_path, s3_prefix):
    for root, dir, files in os.walk('data'):
        for file in files:
            file_path = os.path.join(root, file).replace('\\', '/')
            relpath = os.path.relpath(file_path, directory_path)
            s3_key = os.path.join(s3_prefix, relpath).replace('\\','/')

            s3.upload_file(file_path, bucket_name,  s3_key)  
    print("Directory Uploaded")

upload_directory('data', 's3_data')

Directory Uploaded


Download folder to local system

In [48]:
local_path = 's3_download'
s3_prefix = 's3_data'

def download_dir(local_path, s3_prefix):
    os.makedirs(local_path, exist_ok=True)
    paginator = s3.get_paginator('list_objects_v2')
    for result in paginator.paginate(Bucket=bucket_name, Prefix=s3_prefix):
        if 'Contents' in result:
            for key in result['Contents']:
                s3_key = key['Key']
                
                local_file = os.path.join(local_path, os.path.relpath(s3_key, s3_prefix))
                
                s3.download_file(bucket_name, s3_key, local_file)
    print("Directory Downloaded")

download_dir(local_path, s3_prefix)

Directory Downloaded


Delete all files

In [50]:
def delete_objects():
    response= s3.list_objects_v2(Bucket=bucket_name)
    if 'Contents' in response:
        for obj in response['Contents']:
            s3.delete_object(Bucket=bucket_name, Key=obj['Key'])
    print("Objects Deleted")

delete_objects()

Objects Deleted


In [ ]:
s3.delete_bucket(Bucket=bucket_name)